# SmartLend — TabPFN-2.5 inference server on Colab

TabPFN-2.5 needs more VRAM than the project's local 8 GB RTX 4060 (which caps its in-context training set at 5,000 rows). This notebook hosts it on a **Colab GPU** and exposes a small authenticated HTTP endpoint the local SmartLend backend can call.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is fine).

**Steps** (each cell in order):
1. Install pinned dependencies and the `cloudflared` tunnel binary.
2. Download the **pinned TabPFN-2.5 checkpoint** (a plain `pip install tabpfn` would serve TabPFN-3 — a different model than the project reports).
3. Upload `tabpfn_context.csv` — produce it locally first with `python -m backend.export_tabpfn_context`.
4. Fit the in-context classifier.
5. Start the FastAPI server (in-process).
6. Open the tunnel and copy the printed `SMARTLEND_TABPFN_URL` / `SMARTLEND_TABPFN_TOKEN` into the project's `.env`, then run `python -m backend.run_tabpfn_connection_check` locally.

**Licence:** TabPFN-2.5 outputs are non-commercial (see `models/README.md`). This endpoint is a research/demo instrument; the backend never feeds its scores into a lending decision.

The endpoint lives only while this notebook runs. Free Colab disconnects idle sessions after a while — restart the tunnel cell and update `.env` if the URL changes.

In [ ]:
# 1) Dependencies + tunnel binary -----------------------------------------
# tabpfn>=8,<9 matches the project's local environment; the MODEL version is
# pinned separately by the checkpoint file below, not by the package version.
%pip -q install "tabpfn>=8,<9" fastapi "uvicorn[standard]" pandas scikit-learn
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
# 2) Pinned TabPFN-2.5 checkpoint ------------------------------------------
# Same file and same reasoning as models/README.md: the package default moved
# across releases, so the checkpoint path is what pins v2.5.
import os, urllib.request
CKPT = "/content/tabpfn-v2.5-classifier-v2.5_default.ckpt"
if not os.path.exists(CKPT):
    urllib.request.urlretrieve(
        "https://huggingface.co/Prior-Labs/tabpfn_2_5/resolve/main/tabpfn-v2.5-classifier-v2.5_default.ckpt",
        CKPT,
    )
print(f"checkpoint: {CKPT} ({os.path.getsize(CKPT) / 1e6:.0f} MB)")

In [ ]:
# 3) Upload the in-context training set ------------------------------------
# Produce it locally first:  python -m backend.export_tabpfn_context
# It is the serving 15-feature vocabulary + TARGET (1 = defaulted), built by
# the same customer_profile_service._build_profile derivation the live API
# uses. Home Credit data may not be redistributed, hence upload-at-runtime.
import pandas as pd
from google.colab import files

if not os.path.exists("/content/tabpfn_context.csv"):
    uploaded = files.upload()  # pick models/tabpfn/tabpfn_context.csv
    name = next(iter(uploaded))
    os.rename(name, "/content/tabpfn_context.csv")

context = pd.read_csv("/content/tabpfn_context.csv")
y_context = context.pop("TARGET").astype(int)
FEATURE_NAMES = list(context.columns)
print(f"context: {len(context):,} rows x {len(FEATURE_NAMES)} features, "
      f"default rate {y_context.mean():.4f}")
print(FEATURE_NAMES)

In [ ]:
# 4) Fit the in-context classifier ------------------------------------------
# 'Fit' for TabPFN means loading the context into GPU memory: cost is
# quadratic in context rows, which is exactly why this runs on Colab.
import time
from tabpfn import TabPFNClassifier

clf = TabPFNClassifier(model_path=CKPT, device="cuda")
t0 = time.time()
clf.fit(context.to_numpy(), y_context.to_numpy())
print(f"fitted on {len(context):,} context rows in {time.time() - t0:.1f}s")

# Smoke test: score 5 context rows (values only — proves the forward pass runs).
proba = clf.predict_proba(context.head(5).to_numpy())
print("smoke-test P(default):", [round(float(p), 4) for p in proba[:, 1]])

In [ ]:
# 5) The API server ---------------------------------------------------------
# Wire protocol consumed by backend/app/services/remote_tabpfn_service.py:
#   GET  /health -> model/device/context info
#   POST /score  {"rows": [{feature: value, ...}]} -> {"predictions": [...]}
# Both require the X-SmartLend-Token header printed by the tunnel cell.
import secrets, threading, math
import numpy as np
import tabpfn as tabpfn_pkg
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel

SESSION_TOKEN = os.environ.get("SMARTLEND_TABPFN_TOKEN") or secrets.token_urlsafe(24)
PORT = 8321
MAX_ROWS = 500

# Missing/NaN features are imputed EXACTLY as the serving path does
# (MLPredictor.predict_application): cbes_engine.DEFAULTS.get(col, 0.0).
# The dict is inlined because the backend package is not on Colab.
CBES_DEFAULTS = {
    "credit_score": 0.0, "delinquencies": 10.0, "active_loans": 10.0, "dti": 1.0,
    "employment_tenure_years": 0.0, "annual_income": 1.0,
    "loan_amount": 10_000_000.0, "region": 3.0,
}

app = FastAPI(title="SmartLend remote TabPFN", docs_url=None, redoc_url=None)

def _check_token(request: Request) -> None:
    if request.headers.get("X-SmartLend-Token") != SESSION_TOKEN:
        raise HTTPException(status_code=401, detail="bad or missing X-SmartLend-Token")

class ScoreRequest(BaseModel):
    rows: list[dict]

@app.get("/health")
def health(request: Request) -> dict:
    _check_token(request)
    return {
        "status": "ok",
        "model": "TabPFN-2.5 (pinned checkpoint)",
        "checkpoint": os.path.basename(CKPT),
        "tabpfn_version": tabpfn_pkg.__version__,
        "device": torch.cuda.get_device_name(0),
        "context_rows": int(len(context)),
        "feature_names": FEATURE_NAMES,
        "licence": "TabPFN-2.5 outputs are non-commercial; research/demo use only.",
    }

@app.post("/score")
def score(body: ScoreRequest, request: Request) -> dict:
    _check_token(request)
    if not body.rows:
        return {"predictions": [], "seconds": 0.0}
    if len(body.rows) > MAX_ROWS:
        raise HTTPException(status_code=413, detail=f"max {MAX_ROWS} rows per call")
    frame = pd.DataFrame(body.rows).reindex(columns=FEATURE_NAMES)
    for col in FEATURE_NAMES:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(CBES_DEFAULTS.get(col, 0.0))
    frame = frame.replace([np.inf, -np.inf], 0.0).astype(float)
    t0 = time.time()
    p_default = clf.predict_proba(frame.to_numpy())[:, 1]
    return {
        "predictions": [
            {"p_default": float(p), "p_approve": float(1.0 - p)} for p in p_default
        ],
        "seconds": round(time.time() - t0, 3),
    }

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print(f"server up on 127.0.0.1:{PORT}")
print(f"SESSION TOKEN: {SESSION_TOKEN}")

In [ ]:
# 6) Public tunnel -----------------------------------------------------------
# Cloudflare quick tunnel: no account needed, URL is random per run.
import re, subprocess

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _ in range(120):
    line = proc.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if match:
        public_url = match.group(0)
        break
assert public_url, "tunnel URL not found — rerun this cell"

print("=" * 72)
print("Endpoint is LIVE. In the SmartLend repo, add to .env:")
print()
print(f"SMARTLEND_TABPFN_URL={public_url}")
print(f"SMARTLEND_TABPFN_TOKEN={SESSION_TOKEN}")
print()
print("Then run the connection run locally:")
print("    python -m backend.run_tabpfn_connection_check")
print("and restart the backend to pick up the new env vars.")
print("=" * 72)

## Troubleshooting

- **`No GPU`** — Runtime → Change runtime type → T4 GPU, then rerun from cell 1.
- **CUDA out of memory in cell 4** — re-export a smaller context locally: `python -m backend.export_tabpfn_context --context-rows 5000`, re-upload.
- **Tunnel URL changed after a disconnect** — rerun cell 6, update the two `.env` lines, restart the backend.
- **401 from the backend side** — the token in `.env` does not match this session's printed token.
- **Connection run FAILs on AUC** — almost always a context/feature mismatch: re-run `export_tabpfn_context` and re-upload rather than editing columns by hand.